Einer Barba Abdala 595839

Renata García Morales 612194

Título: Initial implementation of a Deep Learning
pipeline for soccer player and ball detection

Date: 08/05/2026

Materia: Inteligencia Artificial II

*We hereby declare that we have done this activity with academic integrity.*

# **Model selection**

The model selected based on previous research was YOLO26n, belonging to the YOLO family, specifically the v26 Nano variant. This model is single-stage, based on deep convolutional networks, and is capable of predicting bounding boxes and classes at the same time in a single forward pass through the network.

Pretrained weights from the COCO dataset will be used with transfer learning. Specifically, we will start from the checkpoint of the yolo26n.pt and apply fine-tuning on the custom dataset with the 2 classes which will be player and ball. This strategi is considered efficient since the COCO dataset includes the classes person and sports ball and because of this it can be trnasferable to the project's target objects.

The previous research resulted in the conclusion that this model is the best fit for the objectives of the project. The options that were investigated and compared were YOLO26n, YOLOv8 and Faster R-CNN. Because of the previously mentioned single-stage design and the anchor-free approach which means that the model predicts objects locations directly without relying on predifined anchor boxes making it more adaptable to positional changes and scale variations. This makes the model a good option when it comes to the positional changes that can happen because of the drone. It is also the most modern of those evaluated offering not only the computational efficiency essential for this project but it also has good results when it comes to detecting small objects like the ball in this case. At last, the lower parameter count compared to the other models reduces the risk of overfitting when working with a limited dataset which is important given the limited scope of the project.

# **Data preparation**

The aerial video to be analyzed for this class' final project has a duration of 2 minutes and 8 seconds. It is recorded at 30 frames per second, which results in a total of 3,840 frames. Those frames have a resolution of 1280 x 720 pixels.
To properly gather a diverse subset of data to annotate,
sampling is done at a fixed-interval by which one frame is extracted every 20 frames. This will result in a subset of 192 frames distributed evenly throughout the whole video.

This approach offers advantages over others like scene-change detection or motion-based sampling. Scene-change detection, for example, is not suitable in this situation. This is due to the fact that the video always shows the same aerial view of the field, with no scene changes occurring. Apart from that, elements like the bird crossing the frame would incorrectly provoke scene-change events, introducing unwanted samples in the dataset. Fixed-interval sampling guarantees that frames are obtained from the entire video, capturing different moments of player positions and different ball locations.


There are two class labels belonging to this project, each belonging to a type of tracked object. The annotation format must follow YOLO format, being an integer index. Here, 0 is the player class. This class includes any soccer player that is visible in the frame, and those partially covered with shadows. The second class, of index 1, is the ball class. This class incldues the location of the ball at every frame. Due to its size, tighter, more consistent bounding boxes will be prioritized, to ensure better model accuracy.

In [1]:
import cv2
import os
import shutil
 # from google.colab import files
import zipfile
import tempfile
import pandas as pd
from PIL import Image
import yaml
import glob
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
print(os.listdir("dataset"))

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'dataset'

In [10]:
VIDEO_PATH = "2023_05_05_15_02_22-players-and-ball-detection.mp4"

cam = cv2.VideoCapture(VIDEO_PATH)

total_frames = int(cam.get(cv2.CAP_PROP_FRAME_COUNT))
fps          = cam.get(cv2.CAP_PROP_FPS)
width        = int(cam.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cam.get(cv2.CAP_PROP_FRAME_HEIGHT))

cam.release()

In [ ]:
print(total_frames)

0


In [ ]:
SAMPLE_EVERY = 20

cam = cv2.VideoCapture(VIDEO_PATH)

try:
    if not os.path.exists('dataset/frames'):
        os.makedirs('dataset/frames')
except OSError:
    print('Error: Creating directory of data')

currentframe = 0
saved        = 0

while(True):
    ret, frame = cam.read()
    if ret:
        if currentframe % SAMPLE_EVERY == 0:
            name = f'./dataset/frames/frame{currentframe:05d}.jpg'
            print('Creating... ' + name)
            cv2.imwrite(name, frame)
            saved += 1
        currentframe += 1
    else:
                "from pathlib import Path",
        break

cam.release()
cv2.destroyAllWindows()
                "base_dir = Path.cwd()",
                "data[\"train\"] = str(base_dir / \"split/train/images\")",
                "data[\"val\"]   = str(base_dir / \"split/val/images\")",
                "data[\"test\"]  = str(base_dir / \"split/test/images\")",
CLASS_NAMES = {
    0: "player",
                "    yaml.safe_dump(data, f, sort_keys=False)",
}

print("Class labels defined for annotation:")
for idx, name in CLASS_NAMES.items():
    print(f"  {idx} → {name}")

NameError: name 'VIDEO_PATH' is not defined

In [ ]:
shutil.make_archive("frames", 'zip', "dataset/frames")

'/content/frames.zip'

In [ ]:
files.download("frames.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Data labelling**

To assist in the labeling of the 186 frames belonging to the varied subset, the tool Roboflow was chosen.

This web application offers a simplified, effective process to draw bounding boxes over each individual frame. The annotations generated by this application are consistent with the format needed by the YOLO models.

This tool works by allowing the upload of the selected frames. Then, each frame is presented individually to be annotated through bounding boxes, allowing a different type of box to be used depending on class. Afterwards, the annotated frame can be mark as completed and sent to the Annotated Frames section.

This leads to a full Annotated Frames folder containing the full collection of frames, each with 12 annotated players and 1 annotated ball.

This set can then be downloaded, and uploaded to this notebook.

This process begins by loading the exported .zip file directly, and extracting its contents.

In [2]:
with zipfile.ZipFile("SoccerField.yolo26.zip") as zf:
    zf.extractall("dataset")

In [3]:
with open("dataset/data.yaml", "r") as f:
    data = yaml.safe_load(f)

print(data)

{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 2, 'names': ['ball', 'player'], 'roboflow': {'workspace': 'renatas-workspace-bae6f', 'project': 'renatas-workspace-bae6f', 'version': 'dataset', 'license': 'Private', 'url': 'https://app.roboflow.com/renatas-workspace-bae6f/renatas-workspace-bae6f/dataset'}}


Afterwards, the filenames of each individual image can be stored.

In [4]:
images = sorted(glob.glob("dataset/**/images/*.jpg", recursive=True))

With this, there is now a full collection of images loaded. The next step is to manually split these images into the necessary categories.

# **Dataset split**

The images in the annotated subset will be split into a 70-15-15 % train-test-validation segmentation. This will allow for the majority of the images to be used for training, while allowing for sufficient testing and validation.

In [5]:
train, temp = train_test_split(images, test_size=0.3, random_state=42)
val, test   = train_test_split(temp,   test_size=0.5, random_state=42)

In [6]:
for split_name, split_files in [("train", train), ("val", val), ("test", test)]:
    os.makedirs(f"split/{split_name}/images", exist_ok=True)
    os.makedirs(f"split/{split_name}/labels", exist_ok=True)
    for img_path in split_files:
        lbl_path = img_path.replace("images", "labels").replace(".jpg", ".txt")
        shutil.copy(img_path, f"split/{split_name}/images/")
        shutil.copy(lbl_path, f"split/{split_name}/labels/")

Once the files are distributed amongs the different categories, the data.yaml file must be updated to reflect this.

In [7]:
data.update({"train": "split/train/images", "val": "split/val/images", "test": "split/test/images"})
with open("split/data.yaml", "w") as f:
    yaml.dump(data, f)

In [8]:
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 387, Val: 83, Test: 83


# **Model training**

In [2]:
!uv pip install ultralytics
import ultralytics
from ultralytics import YOLO
ultralytics.checks()

Ultralytics 8.4.47  Python-3.12.1 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2080 SUPER, 8192MiB)
Setup complete  (24 CPUs, 31.9 GB RAM, 1146.6/3726.0 GB disk)


In [3]:
# Run inference on an image with YOLO26n
!yolo predict model=yolo26n.pt source='https://ultralytics.com/images/zidane.jpg'

'yolo' is not recognized as an internal or external command,
operable program or batch file.


In [6]:
# Load pretrained YOLO26n checkpoint
model = YOLO("yolo26n.pt")

# Fine-tune on the custom dataset
results = model.train(
    data="split/data.yaml",
    epochs=200,
    imgsz=1280,
    batch=4,
    workers=4,
    amp=True,
    box=12.0,
    cls=3.0,
    dfl=2.0
)

New https://pypi.org/project/ultralytics/8.4.51 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.47  Python-3.12.1 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2080 SUPER, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=12.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=3.0, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=split/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=2.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, na

# **Validation and testing**

In [7]:
import os
import cv2
import numpy as np
from pathlib import Path
from collections import defaultdict

# Load the trained model
model = YOLO("runs/detect/train-4/weights/best.pt")

# Get test images
test_dir = Path("split/test/images")
test_images = sorted(list(test_dir.glob("*.jpg")))[:10]  # First 10 test images

print(f"Testing on {len(test_images)} unseen images from test set...\n")

# Track detections
ball_detections = defaultdict(list)
player_detections = defaultdict(list)
class_names = {0: "player", 1: "ball"}

for img_path in test_images:
    # Run inference
    results = model.predict(source=str(img_path), conf=0.5, verbose=False)
    result = results[0]
    
    img_name = img_path.name
    
    # Extract detections by class
    for box in result.boxes:
        class_id = int(box.cls.item())
        confidence = box.conf.item()
        
        if class_id == 0:  # Player
            player_detections[img_name].append(confidence)
        elif class_id == 1:  # Ball
            ball_detections[img_name].append(confidence)

# Compute statistics
print("=" * 70)
print("BALL DETECTION ACCURACY (Target: 70%+)")
print("=" * 70)
ball_found = sum(1 for dets in ball_detections.values() if dets)
ball_accuracy = (ball_found / len(test_images)) * 100
print(f"Detected ball in {ball_found}/{len(test_images)} frames: {ball_accuracy:.1f}%")

if ball_detections:
    ball_confidences = [conf for dets in ball_detections.values() for conf in dets]
    print(f"Average ball confidence: {np.mean(ball_confidences):.3f}")
    print(f"Median ball confidence: {np.median(ball_confidences):.3f}")
else:
    print("No balls detected - consider increasing training epochs or adjusting conf threshold.")

print("\n" + "=" * 70)
print("PLAYER DETECTION ACCURACY (Expected: 95%+)")
print("=" * 70)
player_found = sum(1 for dets in player_detections.values() if len(dets) > 0)
player_accuracy = (player_found / len(test_images)) * 100
print(f"Detected players in {player_found}/{len(test_images)} frames: {player_accuracy:.1f}%")

if player_detections:
    player_counts = [len(dets) for dets in player_detections.values() if dets]
    if player_counts:
        print(f"Average players per frame: {np.mean(player_counts):.1f} (expected: ~12)")
        print(f"Median players per frame: {np.median(player_counts):.0f}")
        player_confidences = [conf for dets in player_detections.values() for conf in dets]
        print(f"Average player confidence: {np.mean(player_confidences):.3f}")

print("\n" + "=" * 70)
print("RECOMMENDATIONS")
print("=" * 70)
if ball_accuracy < 70:
    print("⚠ Ball accuracy below 70%. Try:")
    print("  1. Increase epochs (currently 50) to 75-100")
    print("  2. Lower conf threshold (currently 0.5) to 0.3-0.4")
    print("  3. Check annotation quality in split/train/labels/")
    print("  4. Verify ball bounding boxes are not too tight")
else:
    print("✓ Ball accuracy target achieved! Consider deploying this model.")

Testing on 10 unseen images from test set...

BALL DETECTION ACCURACY (Target: 70%+)
Detected ball in 10/10 frames: 100.0%
Average ball confidence: 0.791
Median ball confidence: 0.810

PLAYER DETECTION ACCURACY (Expected: 95%+)
Detected players in 2/10 frames: 20.0%
Average players per frame: 1.0 (expected: ~12)
Median players per frame: 1
Average player confidence: 0.692

RECOMMENDATIONS
✓ Ball accuracy target achieved! Consider deploying this model.


## **Accuracy Analysis & Testing on Unseen Data**

After training, test the model on your 10 unseen test frames to measure ball detection accuracy. The optimized training above includes:
- **50 epochs** with early stopping (patience=15) for better convergence  
- **640px images** to preserve small ball details  
- **Aggressive augmentation** to handle shaky footage and lighting changes  
- **Validation enabled** to monitor per-epoch metrics  
- **Lower learning rates** (lr0=0.001, lrf=0.01) for stable fine-tuning  
- **Only 5 frozen layers** to allow network adaptation to the ball domain  

Expected results: **70%+ ball detection accuracy** on unseen test frames, with 95%+ player accuracy.

In [8]:
# Load the best weights saved during training
model = YOLO("runs/detect/train-4/weights/best.pt")

# Evaluate model performance on the validation set
metrics = model.val()

# Report metrics
print(f"mAP50     : {metrics.box.map50:.3f}")
print(f"mAP50-95  : {metrics.box.map:.3f}")
print(f"Precision : {metrics.box.mp:.3f}")
print(f"Recall    : {metrics.box.mr:.3f}")
print(f"Per-class mAP50 : {metrics.box.maps}")

Ultralytics 8.4.47  Python-3.12.1 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2080 SUPER, 8192MiB)
YOLO26n summary (fused): 122 layers, 2,375,226 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 49.72.9 MB/s, size: 306.4 KB)
val: Scanning D:\Repos\FinalProjectAi2\final-project-AI-II\split\val\labels.cache... 83 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 83/83  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.4s/it 8.3s0.5s2
                   all         83       1078      0.949      0.917      0.955      0.529
                  ball         82         82      0.934      0.868      0.942      0.479
                player         83        996      0.964      0.965      0.969      0.578
Speed: 6.6ms preprocess, 11.3ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to D:\Repos\FinalProjectAi2\final-project-AI-II\runs\detect\val
mAP50     : 0.955
mAP50-95  :

In [ ]:
# Frames not used during training
UNSEEN_FRAMES = [10, 410, 810, 1210, 1610,
                 2010, 2410, 2810, 3210, 3610]

os.makedirs("inference_frames", exist_ok=True)

cam          = cv2.VideoCapture(VIDEO_PATH)
currentframe = 0
saved        = 0

while True:
    ret, frame = cam.read()
    if not ret:
        break
    if currentframe in UNSEEN_FRAMES:
        name = f'inference_frames/frame{currentframe:05d}.jpg'
        cv2.imwrite(name, frame)
        saved += 1
    currentframe += 1

cam.release()
cv2.destroyAllWindows()
print(f"Unseen frames saved: {saved}")

In [ ]:
model = YOLO("runs/detect/train-4/weights/best.pt")

inference_imgs = sorted(glob.glob("inference_frames/*.jpg"))

for img_path in inference_imgs:
    results = model(img_path)

    for result in results:
        names = [result.names[cls.item()] for cls in result.boxes.cls.int()]
        confs = result.boxes.conf
        print(f"\n{img_path}")
        print(f"  Detections : {names}")
        print(f"  Confidence : {[round(c.item(), 2) for c in confs]}")

        result.save(filename=img_path.replace("inference_frames", "inference_results"))

    display(IPImage(img_path.replace("inference_frames", "inference_results")))

# **Model saving**

# **Inference demonstration**

# **Referencias**
EXTRACCION DE DATOS:
https://www.geeksforgeeks.org/python/extract-images-from-video-in-python/
https://arxiv.org/html/2408.03340v1

DATA LABELING>
https://docs.ultralytics.com/es/integrations/roboflow/
https://docs.ultralytics.com/es/integrations/roboflow/#roboflow-collect
https://blog.roboflow.com/tips-for-how-to-label-images/


PARA EL MODELO:
https://docs.ultralytics.com/tasks/detect/
https://colab.research.google.com/github/ultralytics/ultralytics/blob/main/examples/tutorial.ipynb#scrollTo=zR9ZbuQCH7FX
https://docs.ultralytics.com/modes/train/
https://docs.ultralytics.com/datasets/detect/coco/